In [1]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from nasch_two_lane import TwoLaneNaSch, TwoLaneNaSchWithLights

plt.rcParams.update({
    "figure.figsize":  (8, 5),
    "font.size":       12,
    "axes.grid":       True,
    "grid.alpha":      0.3,
    "axes.labelsize":  12,
    "axes.titlesize":  13,
    "legend.fontsize": 11,
    "savefig.dpi":     150,
    "savefig.bbox":    "tight",
})

DATA_DIR    = "../data"
FIGURES_DIR = "../figures"
os.makedirs(DATA_DIR,    exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)


def save_results(path, **arrays):
    np.savez(path, **arrays)


def load_or_compute(path, compute_fn, force=False):
    if force or not os.path.exists(path):
        result = compute_fn()
        np.savez(path, **result)
        return result
    with np.load(path) as f:
        return {k: f[k] for k in f.files}

Failed to read module file 'c:\Users\Bakri\AppData\Local\Programs\Python\Python311\Lib\functools.py' for module 'functools': UnicodeDecodeError
Traceback (most recent call last):
  File "C:\Users\Bakri\AppData\Roaming\Python\Python311\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Bakri\AppData\Roaming\Python\Python311\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Bakri\AppData\Local\Programs\Python\Python311\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1204, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1176, in _find_and_load
  File "<frozen importlib._

# Two-lane NaSch with lights — phase offset vs lane-change probability

At fixed moderate inflow (`p_in = 0.4`), with `N = 2` equidistant
traffic lights on an `L = 1000` road, this notebook scans the relative
phase offset `φ` between the two lights against the lane-change
probability `p_chg`. The question is **how the optimal `φ` depends on
`p_chg`** — this is the headline figure for the Option A report
direction.

Three schedules are of particular interest and are marked on every
panel below:

- **in-phase** — `φ = 0` (both lights switch together),
- **anti-phase** — `φ = T_cycle / 2` (opposite switching),
- **green-wave** — `φ = d / v_free`, where `d = L / (N + 1)` is the
  inter-light distance and `v_free` is the **empirically measured**
  free-flow speed, not assumed equal to `v_max`. Dawdling
  (`p_rand = 0.3`) pulls `v_free` below `v_max = 5`, so `φ_gw` is not
  simply `d / v_max`.

The per-light phase is specified through the new `phase_offsets` kwarg
on `TwoLaneNaSchWithLights`: light 0 is fixed at phase 0 and light 1
is swept through `φ`.

Scope: `N = 2` only. Scaling to several lights is deferred.

In [2]:
# Measure the empirical free-flow speed at low density with no lights.
# Using the base class directly is equivalent to TwoLaneNaSchWithLights
# with N=0; neither path has any red-light braking to interfere.
L_ROAD   = 1000
V_MAX    = 5
P_RAND   = 0.3
N_LIGHTS = 2
T_CYCLE  = 60
T_GREEN  = 30

VFREE_P_IN     = 0.1
VFREE_WARMUP   = 1000
VFREE_MEASURE  = 5000
VFREE_SEED     = 0

np.random.seed(VFREE_SEED)
vfree_sim = TwoLaneNaSch(
    L=L_ROAD, n_lanes=2, v_max=V_MAX, p_rand=P_RAND,
    p_chg=1.0, boundary="open", p_in=VFREE_P_IN,
)
vfree_sim.warmup(VFREE_WARMUP)

vel_sum   = 0.0
vel_count = 0
for _ in range(VFREE_MEASURE):
    vfree_sim.step()
    mask = vfree_sim.road >= 0
    if mask.any():
        vel_sum   += float(vfree_sim.road[mask].sum())
        vel_count += int(mask.sum())

v_free     = vel_sum / vel_count
d          = L_ROAD // (N_LIGHTS + 1)
phi_gw_raw = d / v_free
phi_gw     = phi_gw_raw % T_CYCLE

print(f"v_max                 = {V_MAX}")
print(f"v_free (measured)     = {v_free:.3f}")
print(f"d = L // (N+1)        = {d}")
print(f"phi_gw unwrapped      = {phi_gw_raw:.2f}")
print(f"phi_gw mod T_cycle    = {phi_gw:.2f}")

v_max                 = 5
v_free (measured)     = 4.601
d = L // (N+1)        = 333
phi_gw unwrapped      = 72.37
phi_gw mod T_cycle    = 12.37


In [3]:
# Half-size first pass. Full spec in comments; rerun with force=True
# after editing the constants below to get publication-quality curves.
#   Full spec (user-specified): PHI_STEP=2, N_SEEDS=5,
#                               T_WARMUP=5000, T_MEASURE=10000
SWEEP_L         = L_ROAD
SWEEP_V_MAX     = V_MAX
SWEEP_P_RAND    = P_RAND
SWEEP_P_IN      = 0.4
SWEEP_N         = N_LIGHTS
SWEEP_T_CYCLE   = T_CYCLE
SWEEP_T_GREEN   = T_GREEN
SWEEP_PHI_STEP  = 4          # full spec: 2
SWEEP_P_CHG     = np.array([0.0, 0.1, 0.3, 0.5, 1.0])
SWEEP_N_SEEDS   = 3          # full spec: 5
SWEEP_T_WARMUP  = 1000       # full spec: 5000
SWEEP_T_MEASURE = 2000       # full spec: 10000

SWEEP_PHI   = np.arange(0, SWEEP_T_CYCLE + 1, SWEEP_PHI_STEP)
SWEEP_SEEDS = np.arange(SWEEP_N_SEEDS)


def compute_sweep():
    n_phi  = len(SWEEP_PHI)
    n_chg  = len(SWEEP_P_CHG)
    n_seed = len(SWEEP_SEEDS)

    throughput    = np.zeros((n_phi, n_chg, n_seed))
    mean_velocity = np.zeros((n_phi, n_chg, n_seed))
    mean_density  = np.zeros((n_phi, n_chg, n_seed))

    for pi, phi in enumerate(tqdm(SWEEP_PHI, desc="phi")):
        for ci, p_chg in enumerate(SWEEP_P_CHG):
            for si, seed in enumerate(SWEEP_SEEDS):
                np.random.seed(int(seed) * 10_000 + pi * 100 + ci)
                sim = TwoLaneNaSchWithLights(
                    L=SWEEP_L, n_lanes=2,
                    v_max=SWEEP_V_MAX, p_rand=SWEEP_P_RAND,
                    p_chg=float(p_chg), boundary="open",
                    p_in=SWEEP_P_IN,
                    N=SWEEP_N, T_cycle=SWEEP_T_CYCLE, T_green=SWEEP_T_GREEN,
                    phase_offsets=np.array([0, int(phi)]),
                )
                sim.warmup(SWEEP_T_WARMUP)

                vel_sum = 0.0
                car_timesteps = 0
                for _ in range(SWEEP_T_MEASURE):
                    sim.step(record_flow=True)
                    mask = sim.road >= 0
                    n_here = int(mask.sum())
                    if n_here:
                        vel_sum += float(sim.road[mask].sum())
                        car_timesteps += n_here

                throughput[pi, ci, si]    = sim.outflow_count / SWEEP_T_MEASURE
                mean_velocity[pi, ci, si] = vel_sum / max(car_timesteps, 1)
                mean_density[pi, ci, si]  = car_timesteps / (
                    SWEEP_T_MEASURE * 2 * SWEEP_L
                )

    return dict(
        phi_values    = SWEEP_PHI,
        p_chg_values  = SWEEP_P_CHG,
        seeds         = SWEEP_SEEDS,
        throughput    = throughput,
        mean_velocity = mean_velocity,
        mean_density  = mean_density,
        L             = SWEEP_L,
        v_max         = SWEEP_V_MAX,
        p_rand        = SWEEP_P_RAND,
        p_in          = SWEEP_P_IN,
        N             = SWEEP_N,
        T_cycle       = SWEEP_T_CYCLE,
        T_green       = SWEEP_T_GREEN,
        T_warmup      = SWEEP_T_WARMUP,
        T_measure     = SWEEP_T_MEASURE,
    )


sweep = load_or_compute(f"{DATA_DIR}/two_lane_lights_sweep.npz",
                        compute_sweep)
print("sweep arrays:", sorted(sweep.keys()))
print("throughput shape:", sweep["throughput"].shape)

phi:   0%|          | 0/16 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
fig, ax = plt.subplots()

phi_values   = sweep["phi_values"]
p_chg_values = sweep["p_chg_values"]
throughput   = sweep["throughput"]

colours = plt.get_cmap("tab10").colors
for ci, p_chg in enumerate(p_chg_values):
    mean_t = throughput[:, ci, :].mean(axis=-1)
    std_t  = throughput[:, ci, :].std(axis=-1)
    c = colours[ci % len(colours)]
    ax.plot(phi_values, mean_t, color=c, linewidth=1.6,
            label=f"p_chg = {float(p_chg):.1f}")
    ax.fill_between(phi_values, mean_t - std_t, mean_t + std_t,
                    color=c, alpha=0.2)

ymin, ymax = ax.get_ylim()
special = [
    (0.0,                  "in-phase (φ=0)",        "grey"),
    (T_CYCLE / 2,          "anti-phase (φ=T/2)",    "grey"),
    (phi_gw,               f"green-wave (φ={phi_gw:.1f})", "black"),
]
for x, label, c in special:
    ax.axvline(x, color=c, linestyle="--", linewidth=1.2, alpha=0.7)
    ax.text(x, ymax, f" {label}", rotation=90, va="top", ha="left",
            fontsize=9, color=c, alpha=0.9)

ax.set_xlabel("Phase offset φ (timesteps)")
ax.set_ylabel("Throughput (cars/timestep, both lanes)")
ax.set_title("Throughput vs phase offset, two equidistant lights (L=1000, N=2)")
ax.set_xlim(0, T_CYCLE)
ax.legend(loc="best")

fig.savefig(f"{FIGURES_DIR}/02_phi_sweep.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

heat = throughput.mean(axis=-1).T   # shape: (n_p_chg, n_phi)
step = int(phi_values[1] - phi_values[0]) if len(phi_values) > 1 else 1
n_chg = len(p_chg_values)

im = ax.imshow(
    heat,
    extent=[phi_values[0] - step / 2, phi_values[-1] + step / 2,
            -0.5, n_chg - 0.5],
    aspect="auto",
    origin="lower",
    cmap="viridis",
)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Throughput (cars/timestep, both lanes)")

ax.set_yticks(np.arange(n_chg))
ax.set_yticklabels([f"{float(p):.1f}" for p in p_chg_values])
ax.set_ylabel("p_chg")
ax.set_xlabel("Phase offset φ (timesteps)")
ax.set_xlim(phi_values[0] - step / 2, phi_values[-1] + step / 2)
ax.set_title("Throughput over (φ, p_chg)")
ax.grid(False)

for x, label, c in [(0.0, "in-phase", "white"),
                    (T_CYCLE / 2, "anti-phase", "white"),
                    (phi_gw, f"green-wave ({phi_gw:.1f})", "red")]:
    ax.axvline(x, color=c, linestyle="--", linewidth=1.3, alpha=0.9)
    ax.text(x, n_chg - 0.6, f" {label}", rotation=90, va="top",
            ha="left", fontsize=9, color=c)

fig.savefig(f"{FIGURES_DIR}/02_heatmap.png")
plt.show()

## Interpretation

The sweep answers: *given two equidistant lights on an `L = 1000`
road at `p_in = 0.4`, how does the optimal phase offset `φ` depend on
`p_chg`?* Predictions first; observations to be filled after
reviewing the figures produced above.

**Prediction 1 — sharp green-wave peak at low `p_chg`.**
At `p_chg ≈ 0` the two lanes are effectively independent. A platoon
released on green at light 0 can only pass through light 1 without
stopping if light 1 turns green at exactly the right moment, which is
what `φ = φ_gw = d / v_free` encodes. Away from `φ_gw` the platoon
meets a red light and pays a full red-phase penalty, so throughput
should drop visibly.

*Observation:* _(fill in after running: does the `p_chg = 0` curve in
Fig. `02_phi_sweep.png` show a distinct peak at the dashed green-wave
line, with a measurable dip at the in-phase and anti-phase markers?)_

**Prediction 2 — broadening vs sharpening with `p_chg`.**
As `p_chg` rises, a car whose lane happens to stop at a red can in
principle switch to the other lane and continue, which should
**broaden** the green-wave peak (the timing window within which most
cars still escape without stopping widens). The competing story is
that lane-changes add local turbulence and variance; that would
**sharpen** the peak or lower the overall ceiling. Which wins is the
question this figure answers.

*Observation:* _(fill in: at `p_chg = 1`, is the throughput-vs-`φ`
curve visibly flatter/broader around `φ_gw` than at `p_chg = 0`, or
sharper? Does the peak throughput rise, fall, or stay flat as `p_chg`
goes from 0 to 1?)_

**Prediction 3 — anti-phase is poor.**
At `φ = T_cycle / 2` a platoon released from light 0 arrives at light
1 exactly as that light has turned red, which should produce either a
local throughput minimum or a shallow plateau — certainly not a
secondary peak comparable to `φ_gw`.

*Observation:* _(fill in: is `φ = T_cycle/2` a local minimum, a
shallow plateau, or a secondary maximum? Does this behaviour change
with `p_chg`?)_

*If any prediction above is contradicted by the figures, flag it
explicitly here rather than quietly editing the prediction — an
unexpected result is more interesting than a confirmed one.*